In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import spearmanr
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, get_scheduler, DebertaV2Tokenizer
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
# Set device
torch.cuda.set_per_process_memory_fraction(0.7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load data
main_df = pd.read_csv("ranked_responses_final.csv")

In [ ]:
# Normalize ranks
def compute_proportional_ranks(ranks):
    rank_to_weight = {1: 4, 2: 3, 3: 2, 4: 1}
    total_weight = sum(rank_to_weight[r] for r in ranks)
    return [rank_to_weight[r] / total_weight for r in ranks]

model_order = ["openai/gpt-4o", "anthropic/claude-3.5-sonnet", "deepseek/deepseek-chat", "perplexity/sonar"]
data = []
for prompt in main_df["Prompt"].unique():
    sub = main_df[main_df["Prompt"] == prompt]
    models = sub["Model"].tolist()
    ranks = sub["Rank"].tolist()
    if len(models) != 4: continue
    scores = dict(zip(models, compute_proportional_ranks(ranks)))
    if set(scores.keys()) != set(model_order): continue
    source = sub["Source"].iloc[0] if pd.notnull(sub["Source"].iloc[0]) else None
    data.append({"prompt": prompt, "scores": [scores[m] for m in model_order], "source": source})

df_proc = pd.DataFrame(data).dropna(subset=["source"])

In [ ]:
# Category mapping
category_map = {
    'Summaries': ['summaries'],
    'English': ['english_1', 'english_2'],
    'Math': ['math_1', 'math_2'],
    'Coding': ['coding_1', 'coding_2', 'coding_3'],
    'Reasoning': ['reasoning_1', 'reasoning_2', 'reasoning_3']
}

# Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-small")
model_name = "microsoft/deberta-v3-small"

In [ ]:
class PromptDataset(Dataset):
    def __init__(self, prompts, targets, tokenizer, max_length=256):
        self.encodings = tokenizer(prompts, truncation=True, padding="max_length", max_length=max_length)
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return item

class DebertaRegressor(nn.Module):
    def __init__(self, model_name, num_outputs):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Linear(self.backbone.config.hidden_size, num_outputs)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state[:, 0]
        return self.regressor(x)

def convert_to_ranking(scores):
    sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return {model: rank + 1 for rank, (model, _) in enumerate(sorted_models)}

In [ ]:
# === Training per Category ===
for category, sources in category_map.items():
    print(f"\n===== Training for category: {category} =====")
    category_df = df_proc[df_proc["source"].isin(sources)]
    if category_df.empty:
        print(f"No data for category {category}, skipping.")
        continue

    train_df, test_df = train_test_split(category_df, test_size=0.2, random_state=42, stratify=category_df["source"])

    train_ds = PromptDataset(train_df["prompt"].tolist(), train_df["scores"].tolist(), tokenizer)
    test_ds = PromptDataset(test_df["prompt"].tolist(), test_df["scores"].tolist(), tokenizer)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16)

    model = DebertaRegressor(model_name, num_outputs=4).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    epochs = 10
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * epochs)

    best_val_loss = float("inf")
    train_losses, val_losses, spearman_scores = [], [], []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"{category} Epoch {epoch+1} Training"):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            preds = model(input_ids, attention_mask)
            loss = F.mse_loss(preds, labels)
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss, all_preds, all_labels = 0, [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                preds = model(input_ids, attention_mask)
                val_loss += F.mse_loss(preds, labels).item()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(test_loader)
        val_losses.append(avg_val_loss)
        corr = np.mean([spearmanr(p, t).correlation for p, t in zip(all_preds, all_labels)])
        spearman_scores.append(corr)

        print(f"{category} Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Spearman={corr:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"best_deberta_{category}.pt")

    # Plotting
    plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
    plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.title(f"{category} Training Progress")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"training_progress_{category}.png")
    plt.clf()

print("\nAll category models trained and saved.")